# 02 · Evaluate all checkpoints

Rescores every `generator_best.weights.h5` found under `CKPT_ROOT` with one protocol, so
all numbers come from one code path:

1. road score per tile with test-time augmentation over four flips,
2. threshold chosen per model on the **validation** split (best mean buffered IoU), then frozen,
3. 3x3 morphological closing, then the test split is scored once,
4. per-tile buffered IoU (2 px tolerance), strict IoU, relaxed precision and recall, density ratio,
5. arms averaged over seeds and compared tile by tile against the plain U-Net
   (bootstrap 95% interval and Wilcoxon signed-rank test).

Runs whose name starts with `seg_` are rebuilt with the three-class softmax head and scored
from the road probability; everything after the score is identical for both heads.

No retraining. Set `CKPT_ROOT` to the folder holding the run folders.

## 1. Imports

In [ ]:
import csv, json, math, time, random
from pathlib import Path

import numpy as np
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt

from sat2map import models, data, training, evaluation as ev, runs

print("TensorFlow", tf.__version__)
for g in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(g, True)
print("GPUs:", tf.config.list_physical_devices("GPU"))

from collections import defaultdict
from scipy.stats import wilcoxon

In [ ]:
DATA_ROOT = "../data/Dataset"   # folder holding train/ val/ test/
CKPT_ROOT = "../runs"           # searched recursively for generator_best.weights.h5
OUT       = Path("../eval")

SIZE = 256
USE_TTA, USE_CLOSING = True, True
MAIN_ADV, MAIN_RW = 0.3, 10      # loss weights of the main comparison; other runs are ablations
BASELINE_LABEL = "unet"          # reference arm for the paired comparison
BOOT = 2000
OUT.mkdir(parents=True, exist_ok=True)

## 2. Checkpoints

The output head has to come from the run name: an RGB U-Net and a segmentation U-Net have
identical weight shapes, so loading one into the other raises no error and gives silently
wrong numbers.

In [ ]:
VAL_FILES  = data.file_list(DATA_ROOT, "val")
TEST_FILES = data.file_list(DATA_ROOT, "test")
print("val", len(VAL_FILES), "test", len(TEST_FILES))

RUNS = {}
for name, path in runs.find_checkpoints(CKPT_ROOT).items():
    info = runs.parse(name)
    ablation = info["size"] != SIZE or (info["head"] == "rgb" and
                                        (info["adv"], info["rw"]) != (MAIN_ADV, MAIN_RW))
    if ablation:
        print("skipping %s (ablation run, scored in 03_ablations)" % name)
        continue
    RUNS[name] = dict(path=path, **info)
if not RUNS:
    raise SystemExit("No checkpoints found under %s" % CKPT_ROOT)

print("\n%d checkpoints:" % len(RUNS))
print("  %-50s %-16s %-5s %s" % ("run", "arch", "head", "seed"))
for n, d in RUNS.items():
    print("  %-50s %-16s %-5s %s" % (n, d["arch"], d["head"], d["seed"]))

## 3. Target masks (computed once, reused for every run)

In [ ]:
val_ds, test_ds = data.make_ds(VAL_FILES, SIZE), data.make_ds(TEST_FILES, SIZE)
VAL_TAR, TEST_TAR = ev.target_masks(val_ds), ev.target_masks(test_ds)
print("target masks:", VAL_TAR.shape, TEST_TAR.shape)
print("mean road fraction  val %.4f  test %.4f" % (VAL_TAR.mean(), TEST_TAR.mean()))

## 4. Score every checkpoint

One run at a time: predict validation and test, sweep the threshold on validation, score
test once at the frozen threshold and once at a fixed shared threshold for comparison.

In [ ]:
rows, per_image_rows, per_image_iou = [], [], {}

for name, d in RUNS.items():
    head = d["head"]
    print("scoring %-50s (%s, %s)" % (name, d["arch"], head), flush=True)
    gen = models.make_generator(d["arch"], SIZE, head)
    gen.load_weights(d["path"])
    val_score, _  = ev.road_score(gen, val_ds, head, tta=USE_TTA)
    test_score, _ = ev.road_score(gen, test_ds, head, tta=USE_TTA)
    del gen; tf.keras.backend.clear_session()

    best_th, best_val, _ = ev.sweep_threshold(val_score, VAL_TAR, ev.THRESHOLDS[head], USE_CLOSING)
    per, tot = ev.collect(test_score, TEST_TAR, best_th, USE_CLOSING)
    s = ev.summarise(per, tot)
    sF = ev.summarise(*ev.collect(test_score, TEST_TAR, ev.FIXED_THRESH[head], USE_CLOSING))

    label = runs.label_of(d["arch"], head)
    rows.append(dict(run=name, arch=d["arch"], head=head, label=label, seed=d["seed"],
                     val_best_th=best_th, val_iou_perimg=best_val, **s,
                     fixed_th=ev.FIXED_THRESH[head], iou_perimg_fixed=sF["iou_perimg"],
                     iou_pooled_fixed=sF["iou_pooled"]))
    per_image_iou[name] = np.array(per["iou"], float)
    for i, f in enumerate(TEST_FILES):
        per_image_rows.append(dict(run=name, label=label, seed=d["seed"], idx=i,
                                   file=Path(f).name, thresh=best_th,
                                   iou_buffered=per["iou"][i], iou_strict=per["strict"][i],
                                   precision=per["prec"][i], recall=per["rec"][i],
                                   pred_frac=per["frac"][i], true_frac=float(TEST_TAR[i].mean())))
    print("   th %.2f | IoU per-img %.4f  pooled %.4f | prec %.4f  rec %.4f | pred/true %.2f"
          % (best_th, s["iou_perimg"], s["iou_pooled"], s["prec_perimg"],
             s["rec_perimg"], s["density_ratio"]))

def write_csv(path, rows_):
    with open(path, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows_[0].keys()))
        w.writeheader(); w.writerows(rows_)

write_csv(OUT / "final_eval.csv", rows)
write_csv(OUT / "per_image_eval.csv", per_image_rows)
print("\nwrote final_eval.csv and per_image_eval.csv to", OUT)

## 5. Mean over seeds per architecture

Read the spread column alongside any gap between arms: a gap smaller than the seed spread is not a result.

In [ ]:
by_label = defaultdict(list)
for r in rows:
    by_label[r["label"]].append(r)
order = [l for l in ["unet", "unet_bottleneck", "unet_mid", "resnet9", "attn_unet"] if l in by_label]
order += sorted(l for l in by_label if l not in order)

hdr = "%-18s %3s %11s %11s %10s %8s %8s %9s %8s" % (
    "arm", "n", "IoU per-img", "IoU pooled", "strict", "prec", "recall", "pred/true", "spread")
print(hdr); print("-" * len(hdr))
agg = []
for a in order:
    rs = by_label[a]
    m = lambda k: float(np.mean([r[k] for r in rs]))
    vals = [r["iou_perimg"] for r in rs]
    print("%-18s %3d %11.4f %11.4f %10.4f %8.4f %8.4f %9.2f %8.4f"
          % (a, len(rs), m("iou_perimg"), m("iou_pooled"), m("strict_perimg"),
             m("prec_perimg"), m("rec_perimg"), m("density_ratio"), max(vals) - min(vals)))
    agg.append(dict(arm=a, n_seeds=len(rs), seed_runs=" ".join("%.4f" % v for v in vals),
                    iou_perimg=m("iou_perimg"), iou_pooled=m("iou_pooled"),
                    strict_perimg=m("strict_perimg"), precision=m("prec_perimg"),
                    recall=m("rec_perimg"), density_ratio=m("density_ratio"),
                    spread=max(vals) - min(vals)))
write_csv(OUT / "by_arch.csv", agg)

## 6. Paired comparison against the U-Net

Every arm is scored on the same 300 test tiles, so arms are compared tile by tile. Per-tile IoU
is averaged over seeds, the baseline is subtracted, and tiles are resampled 2,000 times for a
95% interval. `*` marks an interval that excludes zero. `seeds` says whether both runs of the arm
sit above (or below) both runs of the baseline.

In [ ]:
rng = np.random.default_rng(0)
arm_iou = {lab: np.nanmean(np.stack([per_image_iou[r["run"]] for r in rs]), axis=0)
           for lab, rs in by_label.items()}
base = arm_iou[BASELINE_LABEL]
base_runs = [r["iou_perimg"] for r in by_label[BASELINE_LABEL]]

hdr = "%-18s %9s %9s %9s %8s %11s  %s" % ("arm", "mean diff", "CI low", "CI high",
                                         "better%", "Wilcoxon p", "seeds")
print("per-tile buffered IoU minus %s" % BASELINE_LABEL); print(hdr); print("-" * len(hdr))
comp = []
for a in order:
    if a == BASELINE_LABEL:
        continue
    b = ev.paired_bootstrap(arm_iou[a], base, BOOT, rng)
    p = float(wilcoxon(b["diffs"]).pvalue)
    mine = [r["iou_perimg"] for r in by_label[a]]
    seeds = ("both above" if min(mine) > max(base_runs) else
             "both below" if max(mine) < min(base_runs) else "overlap")
    flag = "" if b["ci_low"] <= 0 <= b["ci_high"] else "  *"
    print("%-18s %9.4f %9.4f %9.4f %7.1f%% %11.2e  %s%s" % (
        a, b["mean_diff"], b["ci_low"], b["ci_high"], 100 * b["frac_better"], p, seeds, flag))
    comp.append(dict(arm=a, baseline=BASELINE_LABEL, n_tiles=b["n_tiles"],
                     mean_diff=b["mean_diff"], ci_low=b["ci_low"], ci_high=b["ci_high"],
                     frac_better=b["frac_better"], wilcoxon_p=p, seeds=seeds,
                     excludes_zero=not (b["ci_low"] <= 0 <= b["ci_high"])))
write_csv(OUT / "paired_vs_baseline.csv", comp)

## 7. IoU by road density

Splits the test tiles into quartiles of true road fraction, which shows where each arm earns
or loses its score (and why per-tile and pooled IoU disagree).

In [ ]:
true_frac = TEST_TAR.reshape(TEST_TAR.shape[0], -1).mean(axis=1)
bins = np.digitize(true_frac, np.quantile(true_frac, [0.25, 0.50, 0.75]))
qnames = ["Q1 sparsest", "Q2", "Q3", "Q4 densest"]

def quartile_means(v):
    return [v[(bins == q) & ~np.isnan(v)].mean() for q in range(4)]

print("%-18s " % "arm" + " ".join("%11s" % q for q in qnames))
for a in order:
    print("%-18s " % a + " ".join("%11.4f" % x for x in quartile_means(arm_iou[a])))
print("mean road fraction per quartile:",
      " ".join("%.4f" % true_frac[bins == q].mean() for q in range(4)))

fig, ax = plt.subplots(figsize=(7, 4.5))
for a in order:
    ax.plot(range(4), quartile_means(arm_iou[a]), marker="o", label=a)
ax.set_xticks(range(4)); ax.set_xticklabels(qnames)
ax.set_ylabel("per-tile buffered IoU"); ax.set_xlabel("target road density")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT / "iou_by_density.png", dpi=130); plt.show()

## 8. Qualitative samples

Saves, for four random road tiles plus each arm's three best and three worst tiles, the satellite
input, target, every run's predicted map and its thresholded mask, under `OUT/qual/`.

In [ ]:
N_RANDOM, N_BEST, N_WORST, SAMPLE_SEED = 4, 3, 3, 1907096
PALETTE = np.array([[208, 204, 196], [255, 255, 255], [158, 203, 232]], np.uint8)  # bg, road, water
SDIR = OUT / "qual"; SDIR.mkdir(exist_ok=True)

road_tiles = np.where(true_frac > 0)[0]
random_idx = sorted(np.random.default_rng(SAMPLE_SEED).choice(road_tiles, N_RANDOM, replace=False).tolist())
wanted = set(random_idx)
for lab, v in arm_iou.items():
    ok = np.array([i for i in road_tiles if not np.isnan(v[i])])
    srt = ok[np.argsort(v[ok])]
    wanted.update(srt[:N_WORST].tolist()); wanted.update(srt[-N_BEST:].tolist())
SEL = sorted(wanted)

def save_rgb(img, path):
    cv2.imwrite(str(path), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

sel_ds = data.make_ds([TEST_FILES[i] for i in SEL], SIZE)
for j, (src, tar) in enumerate(sel_ds):
    i = SEL[j]
    for kind, t in (("sat", src), ("target", tar)):
        save_rgb(np.clip((t[0].numpy() * 0.5 + 0.5) * 255, 0, 255).astype(np.uint8),
                 SDIR / ("%03d_%s.png" % (i, kind)))

th_of = {r["run"]: r["val_best_th"] for r in rows}
for name, d in RUNS.items():
    gen = models.make_generator(d["arch"], SIZE, d["head"])
    gen.load_weights(d["path"])
    rdir = SDIR / name; rdir.mkdir(exist_ok=True)
    for j, (src, _) in enumerate(sel_ds):
        o = ev.forward(gen, src, USE_TTA)[0].numpy()
        if d["head"] == "seg":
            img, score = PALETTE[np.argmax(o, axis=-1)], o[..., models.ROAD]
        else:
            o01 = o * 0.5 + 0.5
            img, score = np.clip(o01 * 255, 0, 255).astype(np.uint8), o01.min(axis=-1)
        save_rgb(img, rdir / ("%03d_map.png" % SEL[j]))
        cv2.imwrite(str(rdir / ("%03d_mask.png" % SEL[j])),
                    ev.pred_mask(score, th_of[name], USE_CLOSING).astype(np.uint8) * 255)
    del gen; tf.keras.backend.clear_session()
print("saved %d tiles for %d runs to %s" % (len(SEL), len(RUNS), SDIR))